<a href="https://colab.research.google.com/github/piyushnanwani/RAG-system-building-challenge/blob/main/3_May_RAGSystemBuildingChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U google-generativeai chromadb sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [11]:
from google import genai
from google.genai import types

# 1. Initialize the Client
client = genai.Client(api_key="")

# 2. Use the correct Preview model ID
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    config=types.GenerateContentConfig(
        temperature=0.1,
        top_p=0.95
    ),
    contents="Testing connection: Respond with 'System Ready'"
)

print(response.text)

System Ready


In [13]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb

# 1. Load and Chunk with Metadata
reader = PdfReader("SBI_Contract.pdf")
chunks = []
metadatas = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    # Splitting pages into 2 halves for better search granularity
    mid = len(text) // 2
    chunks.extend([text[:mid], text[mid:]])
    metadatas.extend([{"page": i+1}, {"page": i+1}])

# 2. Initialize 2026 Persistent Database
# Using PersistentClient so the data stays in the /content/ folder
chroma_client = chromadb.PersistentClient(path="./sbi_contract_db")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

collection = chroma_client.get_or_create_collection(name="sbi_it_contract")

# 3. Index with Metadata
for i, chunk in enumerate(chunks):
    vector = embed_model.encode(chunk).tolist()
    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[vector],
        documents=[chunk],
        metadatas=[metadatas[i]]
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
def query_contract(user_query):
    # 1. Similarity Search (Fetch top 5 results for complex contracts)
    query_vector = embed_model.encode(user_query).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=5)

    # 2. Format Context with Page Numbers
    context_list = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        context_list.append(f"[Page {meta['page']}]: {doc}")

    context = "\n---\n".join(context_list)

    # 3. Execution
    full_prompt = f"Using the following sections from the SBI Contract, answer: {user_query}\n\nCONTEXT:\n{context}"

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        config=legal_config,
        contents=full_prompt
    )
    return response.text

# Test a high-stakes question
print(query_contract("What are the specific financial penalties for a Tier 1 system failure?"))

Based on the provided contract clauses, the term **"Tier 1 system failure" does not appear**. The contract instead categorizes system issues and performance failures using "Severity" levels and "Uptime Metrics."

The specific financial penalties for the highest levels of failure defined in the text are as follows:

**1. Severity-Based Penalties (Page 48)**
*   **Critical Severity:** 2% of the cost of Quarterly maintenance charges.
*   **Major Severity:** 1.5% of the cost of Quarterly maintenance charges.

**2. Uptime/Availability Penalties (Page 38)**
If the service provider fails to maintain the guaranteed uptime of 99.9% on a quarterly basis, the following penalties apply:
*   **Availability >= 99.0% and < 99.5%:** 2% of the cost of Quarterly maintenance charges.
*   **Availability >= 98.5% and < 99.0%:** Penalty at an incremental rate of 1% (in addition to a base of 2%) of the cost of Quarterly maintenance charges.

**3. Help Desk Non-Performance (Page 48)**
*   **Non-availability o

In [16]:
# Configuration for Legal Accuracy
legal_config = types.GenerateContentConfig(
    # REMOVED: model="gemini-3-flash-preview",
    temperature=0.0,  # CRITICAL: 0.0 for zero creativity in legal work
    top_p=1,
    system_instruction="You are a senior legal auditor. Answer based ONLY on the provided contract clauses."
)